In [0]:
#set file paths here
path_departments    = '/departments.csv'     #paste_file_path_inside_quotations
path_employees      = '/employees.csv'       #paste_file_path_inside_quotations
path_sales          = '/sales.csv'           #paste_file_path_inside_quotations
path_orders         = '/orders.json'         #paste_file_path_inside_quotations
path_server_metrics = '/server_metrics.json' #paste_file_path_inside_quotations

In [0]:
# import necessary libraries

from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, MapType, DecimalType
from pyspark.sql.functions import col, to_date, avg, max as max_s, min as min_s, count, rank, dense_rank, row_number, sum as sum_s, lag, stack, expr, year, month, dayofmonth, explode, explode_outer
from pyspark.sql.window import Window

**BASIC PYSPARK**

**Q1**
Read Employees as a DataFrame.

printSchema()

show()

inferSchema=False then manually define schema.


In [0]:
employees_df = spark.read\
                    .format('csv')\
                    .option('header','true')\
                    .load(path_employees)

employees_df.printSchema()
employees_df.show()

employees_schema = StructType([StructField('emp_id'    , IntegerType(), nullable = True, metadata = None)
                              ,StructField('name'      , StringType() , nullable = True, metadata = None)
                              ,StructField('dept_id'   , IntegerType(), nullable = True, metadata = None)
                              ,StructField('salary'    , StringType() , nullable = True, metadata = None)
                              ,StructField('city'      , StringType() , nullable = True, metadata = None)
                              ,StructField('join_date' , StringType() , nullable = True, metadata = None)
                              ,StructField('manager_id', IntegerType(), nullable = True, metadata = None)
                   ])
employees_df = spark.read\
                    .format('csv')\
                    .option('header','true')\
                    .option('inferSchema','false')\
                    .schema(employees_schema)\
                    .load(path_employees)
# DateType() for join_date is not able to read the schema and is showing up nulls in the column, it also doesn't any format parameter, its only useful when you use a parquet file as a source
# i am explicitly defining a column with date datatype instead of string, THIS STEP CAN BE DONE LATER though
employees_df = employees_df.withColumn('join_date'
                                       ,to_date(col('join_date'),'dd-MM-yyyy')
                            )
display(employees_df)


------------------------------------------------------------------------------------------------------------------

**Q2**
Select

emp_id
name
salary

only.

In [0]:
employees_df_q2 = employees_df.select('emp_id'
                                     ,'name'
                                     ,'salary'
                               )
display(employees_df_q2)

**Q3**
Filter

Salary > 80,000

In [0]:
employees_df_q3 = employees_df.filter(col("salary")>80000
                               )
display(employees_df_q3)

**Q4**
Find employees

Chicago
AND
salary > 70000

In [0]:
employees_df_q4 = employees_df.filter(  (col('city') == 'Chicago')
                                      & (col('salary') > 70000)
                               )

display(employees_df_q4)

**Q5**
Create

Annual Salary

salary * 12

In [0]:
employees_df_q5 = employees_df.withColumn('annual_salary', col('salary') * 12 
                               )
display(employees_df_q5)

**Q6**
Rename

manager_id to manager

In [0]:
employees_df_q6 = employees_df.withColumnRenamed('manager_id','manager')
display(employees_df_q6)

**AGGREGATIONS**

**Q7**
Department wise

Average salary

In [0]:
employees_df_q7 = employees_df.groupBy(col("dept_id")).agg(avg(col('salary')))

display(employees_df_q7)

**Q8**
Department wise

Maximum salary

Minimum salary

Average salary

Employee count

In [0]:
employees_df_q8 = employees_df.groupBy(col('dept_id')).agg(max_s(col('salary'))
                                                          ,min_s(col('salary'))
                                                          ,avg(col('salary'))
                                                          ,count(col('emp_id'))
                                                       )
display(employees_df_q8)

**Q9**
Find top 3 highest paid employees.

In [0]:
employees_df_q9 = employees_df.select('emp_id').orderBy(col('salary').desc()).limit(3)
display(employees_df_q9)

**Q10**
Count employees city-wise.

In [0]:
employees_df_q10 = employees_df.groupBy(col('city')).agg(count(col('emp_id')))
display(employees_df_q10)

**Q11**
Find departments having Average salary > 75,000

In [0]:
employees_df_q11 = employees_df.groupBy(col('dept_id')).agg(avg(col('salary')).alias('average_salary')).filter(col('average_salary')>75000)
display(employees_df_q11)

**JOINS**

**Q12**
Join Employees with Departments.

In [0]:
departments_df = spark.read.format('csv')\
                           .option('header', 'true')\
                           .load(path_departments)

display(departments_df)

emp_dept_joined_df_q12 = employees_df.join(departments_df
                                          ,employees_df.dept_id == departments_df.dept_id
                                          ,'left'
                                      )\
                                     .drop(departments_df.dept_id) #to remove an extra dept_id column coming from departments dataframe.
display(emp_dept_joined_df_q12)

**Q13** Find employees without matching department.

In [0]:
emp_dept_joined_df_q13 = employees_df.join(departments_df
                                          ,employees_df.dept_id == departments_df.dept_id
                                          ,'left_anti'
                                      )\
                                     .drop(departments_df.dept_id)    
display(emp_dept_joined_df_q13) #all employees have matching employees, no rows returned

**Q14** Find departments without employees.

In [0]:
emp_dept_joined_df_q14 = employees_df.join(departments_df
                                          ,employees_df.dept_id == departments_df.dept_id
                                          ,'left'  
                                      )\
                                     .drop(departments_df.dept_id)
emp_dept_joined_df_q14 = emp_dept_joined_df_q14.groupBy(col('dept_name')).agg(count(col('emp_id')).alias('count_of_employees'))

display(emp_dept_joined_df_q14.filter(col('count_of_employees')==0)) #all departments have atleast 1 employee, no rows returned

**WINDOW FUNCTIONS**

**Q16**
Rank employees by salary.
Use

rank

dense_rank

row_number

In [0]:
emp_df_ranked_df_q17 = employees_df.withColumn('rank_col'
                                              ,rank().over(Window.orderBy(col('salary').desc()))
                                    )
display(emp_df_ranked_df_q17)

emp_df_dense_ranked_df_q17 = employees_df.withColumn('rank_col'
                                                    ,rank().over(Window.orderBy(col('salary').desc()))
                                          )
display(emp_df_dense_ranked_df_q17)

emp_df_row_numbered_df_q17 = employees_df.withColumn('rank_col'
                                                    ,row_number().over(Window.orderBy(col('salary').desc()))
                                          )
display(emp_df_row_numbered_df_q17)

**Q17**
Find highest salary employee in each department.

In [0]:
emp_dept_joined_df_q17 = employees_df.join(departments_df
                                          ,employees_df.dept_id == departments_df.dept_id
                                          ,'left'
                                      )\
                                     .drop(departments_df.dept_id)
display(emp_dept_joined_df_q17.groupBy(col('dept_name')).agg(max_s(col('salary'))))

**Q18**
Find second highest salary in every department.

In [0]:
emp_dept_joined_df_q18 = employees_df.join(departments_df
                                          ,employees_df.dept_id == departments_df.dept_id
                                          ,'left' 
                                      )\
                                     .drop(departments_df.dept_id)
                                     
emp_dept_joined_df_q18 = emp_dept_joined_df_q18.withColumn('salary_rank'
                                                          ,dense_rank().over(Window.partitionBy(col('dept_name')).orderBy(col('salary').desc()))
                                                )
display(emp_dept_joined_df_q18.filter(col('salary_rank')==2).select('dept_name', 'salary'))

**Q19**
Running total salary ordered by join date.

In [0]:
employees_df_q19 = employees_df.withColumn('running_total_salary'
                                          ,sum_s(col('salary')).over(Window.orderBy(col('join_date')).rowsBetween(Window.unboundedPreceding,0))
                                )
display(employees_df_q19.select('salary', 'join_date', 'running_total_salary'))

**Q20**
Lag and Lead

Find salary difference with previous employee.

In [0]:
employees_df_q20 = employees_df.withColumn('salary_difference'
                                          ,col('salary') - lag(col('salary'), 1 , 0).over(Window.orderBy(col('join_date'))).cast(IntegerType())
                                )  
display(employees_df_q20.select('emp_id'
                               ,'salary'
                               ,'salary_difference' 
                         )  
)

**Q21**
Moving average salary over previous 3 employees.

In [0]:
employees_df_q21 = employees_df.withColumn('moving_average'
                                          ,avg(col('salary')).over(Window.orderBy(col('join_date')).rowsBetween(-2, Window.currentRow))
                                )
display(employees_df_q21)

**Q22** Pivot

Region vs Total Sales

In [0]:
sales_df = spark.read.format('csv')\
                     .option('header','true')\
                     .load(path_sales)
emp_sls_joined_df_q22 = employees_df.join(sales_df
                                         ,employees_df.emp_id == sales_df.emp_id
                                         ,'left'
                                     )\
                                    .drop(sales_df.emp_id)
emp_sls_joined_df_q22 = emp_sls_joined_df_q22.selectExpr('emp_id AS emp_id'
                                                        ,'CAST(amount as INT) AS amount'
                                                        ,'region as region'
                                              )

emp_sls_joined_df_q22 = emp_sls_joined_df_q22.groupBy().pivot('region').sum('amount')

display(emp_sls_joined_df_q22)

**Q23**
Unpivot using stack()

In [0]:
emp_pivoted_df_q23 = emp_sls_joined_df_q22.select(expr("stack(4, 'East', East, 'North', North, 'South', South, 'West', West) as (region, sales)"))
display(emp_pivoted_df_q23)

**Q24**
Split join_date into

Year

Month

Day

In [0]:
employees_df_q23 = employees_df.withColumn('year_join_date'
                                          ,year(col('join_date'))
                                )\
                               .withColumn('month_join_date'
                                          ,month(col('join_date'))  
                                )\
                               .withColumn('date_join_date'
                                          ,dayofmonth(col('join_date'))  
                                )
display(employees_df_q23)

**Q25** & **Q26**

Flatten nested JSON and 

Explode an array column.in orders.json

In [0]:
orders_df = spark.read.format('json')\
                      .option('multiline','true')\
                      .load(path_orders)\

orders_df = orders_df.select(col('customer.address')
                            ,col('customer.id')
                            ,col('customer.name')
                            ,explode(col('items')).alias('items')
                      )\
                     .select(col('address.city')
                            ,col('address.pincode')
                            ,col('address.state')
                            ,col('id')
                            ,col('name')
                            ,col('items.item_id')
                            ,col('items.pricing.tax')
                            ,col('items.pricing.unit_price')
                            ,col('items.product_name')
                            ,col('items.quantity')
                      )
                     
display(orders_df)

**Q27**
Explode a map column.

In [0]:
server_metrics_df = spark.read.format('json')\
                              .option('multiline','true')\
                              .load(path_server_metrics)
display(server_metrics_df)

server_metrics_schema = StructType([StructField('location'      , StringType()                       , nullable = True, metadata = False)
                                   ,StructField('server_id'     , StringType()                       , nullable = True, metadata = False)
                                   ,StructField('cpu_usage_pct' , MapType(StringType(),DecimalType()), nullable = True, metadata = False)
                                   ,StructField('service_status', MapType(StringType(), StringType()), nullable = True, metadata = False)
                        ])

server_metrics_df = spark.read.format('json')\
                              .schema(server_metrics_schema)\
                              .option('multiline','true')\
                              .load(path_server_metrics)
server_metrics_df = server_metrics_df.select('location'
                                            ,'server_id'
                                            ,explode_outer("cpu_usage_pct").alias('core_number','clock_pct') #not using explode because it removes the null row in cpu_usage_pct
                                            ,explode_outer("service_status").alias('docker', 'status')
                                      )
display(server_metrics_df)

**Q28**
Dataset has 500 million rows. How would you optimize?
Expected discussion


Predicate Pushdown - Spark sends a filter to the source database(Delta/Parquet/Iceberg) itself. Instead of Reading all the 500 million rows into the RAM of cluster, spark sends a filter to source database to reduce the read and processing time.

Partition Pruning - A step further from Predicate Pushdown, when a table is patitioned on disk by a specific column, spark skips reading the files where the filter passed is not applicable. for an example - the table is partitioned by month-year column, when a filter operation is called filter(col('date'))=='2021-07-01', spark will physically only read the partition /year=2021/month=07 and will skip the other folders reducing the time required to read the entire table/rows/files.

Delta - Delta Lake is an open-source storage layer built on top of Parquet that brings ACID transactions and advanced indexing mechanism to big data. OPTIMIZE & File Compaction: Consolidates thousands of small files into optimal target sizes (~1 GB per file), reducing file-system listing overhead. Z-Ordering (Multi-dimensional Clustering): Colocates related information in the same set of files using Z-values. Running OPTIMIZE table ZORDER BY (customer_id, region) allows Spark to skip files based on non-partition columns. Data Skipping Metadata: Maintains detailed stats (min/max values per column per file) in the _delta_log transaction log, making file skipping faster than standard Parquet.

Broadcast - When joining a massive 500M-row fact table with a small dimension table (e.g., a lookup table of 10,000 store locations), Spark's default behavior is a Sort-Merge Join, which requires shuffling (moving) the 500M rows across the network between cluster nodes. A Broadcast Join copies the small table in its entirety to every executor node. The 500M-row dataset stays right where it is, eliminating network shuffle entirely. df_joined = df_fact_500m.join(broadcast(df_dim_small), "store_id")

AQE - Adaptive Query Executioner, enabled by default in modern Spark (v3.0+), re-optimizes physical query execution plans at runtime using real-time stage statistics. Dynamically Coalescing Shuffle Partitions: Reduces the default 200 shuffle partitions down to a smaller, optimal number if filtered data is tiny, preventing small-file overhead. Dynamically Converting Sort-Merge Join to Broadcast Join: If a filter reduces a 10M row table down to 5 MB mid-query, AQE switches the join strategy to Broadcast on the fly. Handling Data Skew: Automatically splits heavily skewed shuffle partitions into smaller sub-partitions so a single node doesn't hang at 99% progress.

cache & persist - cache(): Stores a DataFrame in memory (MEMORY_AND_DISK storage level by default).
persist(): Offers fine-grained control over storage levels allowing us to store data in MEMORY_ONLY, MEMORY_AND_DISK_SER (serialized), DISK_ONLY, or replicated across multiple nodes.

Photon - Photon is Databricks' vectorized C++ engine designed from the ground up to replace the traditional Java/Scala JVM execution engine. It accelerates Spark workloads directly at the hardware layer by taking advantage of instruction-level parallelism (SIMD). 500 million rows usually strain CPU resources during string parsing, regex, mathematical transformations, aggregations, and hash joins. Enabling Photon offloads these heavy CPU operations out of the JVM into optimized native code, providing a 2x to 5x speedup on end-to-end processing of large datasets without requiring any change to your PySpark code.

**Q29**
Broadcast Join - When should it be used?

How large should the lookup table be? How to force broadcast?


Ans - Broadcast join is used to avoid shuffling in between cluster nodes. Broadcast join copies the smaller table (100 megs at max) to every node so that rows do not have to be shuffled for the join to happen.
The lookup table can be at max of 8 Gigabytes by theory but the default broadcast join threshold is only 10 Megabytes. It is recommended on some websites that we can broadcast upto a small table of upto 100 megabytes.

_FORCE BROADCAST JOIN_

from pyspark.sql.functions import broadcast

result_df = large_df.join(broadcast(small_df), "join_key")


**Q30** Repartition vs Coalesce. When to use each?

Both are used to change the number of partitions of the data being processed.

Repartition - You can increase as well as decrease the number of partitions. Repartition makes the partitions evenly balanced but can also cause a full row shuffle across the nodes. Because of this repartition requires high compute power.

Coalesce - You can only decrease the number of partitions. Coalesce does not make the partitions evenly balanced, they might be skewed depending on how the data is. Coalesce demands lower compute power when compared to Repartition.

**Q31**
Explain
narrow vs wide transformations. Give examples.

Any transformations that requires data movement across the network (or commonly called as "shuffle") is termed as a wide transforamtion. The rest are narrow transformations.

Narrow Transformations  - [map(), filter(), flatMap(), union(), sample(), withColumn()] | Wide Transformations [groupBy(), join(), distinct(), orderBy(), repartition()]

1. Do not cause data shuffles.                                                          | They cause data shuffles

2. each partition maps to at most one output partition.                                 | data from one(or many) partitions can be split to multiple partitions, its a many to many relationship

3. Narrow Transformations always execute within the same stage.                         | Wide Transformations create a new stage in the DAG

4. Narrow Transformations support task chaining.                                        | Not supported in Wide transformations, must wait for shuffle/read/write

5. Narrow Transformations are fast.                                                     | Wide Transformations are expensive for the network.



**Q32** Why does shuffle happen? How to reduce shuffle?

A shuffle is a process of redistributing data across the executor nodes so rows sharing same key end up in same partitions. Whenever a wide transformation like - [groupBy(), join(), distinct(), orderBy(), repartition()] is called in a shuffle is triggered (Shuffle is not necessarily triggered each and every time because, depends on the data being processed in the executors, but most of the times a shuffle is triggered).

Shuffle is an expensive operation -
1. Shuffle Write (Disk I/O): Worker nodes process local data, serialize the intermediate results, and write them to local disk files.
2. Network Fetch (Network I/O): Receiving nodes make network requests to pull their designated data blocks across the cluster network.
3. Shuffle Read & Merge (Memory/CPU): Receiving nodes deserialize the incoming data, sort or hash-group it in memory (or spill to disk if memory is exhausted), and perform the final computation.